[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_he_en_colab_validation.ipynb)

Open this exact notebook in Colab, connect to a runtime, and run cells from the top using ▶. GitHub Preview displays the notebook without executing it.


# LaBSE — Hebrew–English employer-name equivalence probe

**Purpose:** Test whether the *pretrained* LaBSE model assigns higher cross-lingual similarity to corresponding organization names than to unrelated organizations. No retraining, paid API, token, GPU, or manually maintained dictionary is required. Run every cell in **Google Colab (free CPU)**.

**Limitations:** LaBSE is trained for cross-language sentence representations, **not** verified employer entity resolution. This notebook reports observed cosine similarities and nearest neighbors. A high score is not proof that names refer to the same legal entity. Real data integration requires validation. The first execution downloads publicly available model weights (~1.9 GB) and requires Internet access. No model inference result is claimed until this notebook executes.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

import torch
from sentence_transformers import SentenceTransformer
import numpy as np

print("PyTorch:", torch.__version__)
print("Device: CPU (no GPU required)")

In [ ]:
from IPython.display import HTML, display
from html import escape
import numpy as np

# Data examples are the original test inputs. Extra Hebrew forms below are
# diagnostic candidates, NOT additional observed values from the survey.
original_labels = [
    "Intel", "אינטל", "NVIDIA", "נבידיה",
    "Microsoft", "מיקרוסופט", "bird", "birds", "ציפור",
    "קרית גת", "קריית גת", "בית ספר יסודי", "primary school"
]
alternate_nvidia_spellings = ["אנבידיה", "אנווידיה"]
labels = original_labels + alternate_nvidia_spellings

torch.set_num_threads(2)
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
vectors = model.encode(
    labels, batch_size=4, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False
)
assert vectors.shape == (len(labels), 768), vectors.shape
assert np.isfinite(vectors).all()
positions = {label: i for i, label in enumerate(labels)}

def cosine(a, b):
    return float(vectors[positions[a]] @ vectors[positions[b]])

def show_table(heading, rows, caption=None):
    html_rows = []
    for left, right, description in rows:
        score = cosine(left, right)
        html_rows.append(
            "<tr>"
            f"<td>{escape(left)}</td><td>{escape(right)}</td>"
            f"<td style='text-align:right;font-variant-numeric:tabular-nums'>{score:.4f}</td>"
            f"<td>{escape(description)}</td>"
            "</tr>"
        )
    table = (
        f"<h3>{escape(heading)}</h3>"
        "<table style='border-collapse:collapse;width:100%;font-size:14px'>"
        "<thead><tr>"
        "<th style='text-align:left'>Name A</th><th style='text-align:left'>Name B</th>"
        "<th style='text-align:right'>Cosine similarity</th>"
        "<th style='text-align:left'>Known relationship</th>"
        "</tr></thead><tbody>"
        + "".join(html_rows) +
        "</tbody></table>"
    )
    if caption:
        table += f"<p><small>{escape(caption)}</small></p>"
    display(HTML(table))

display(HTML(
    "<h2>LaBSE: cross-language name matching</h2>"
    f"<p>Successfully encoded <b>{len(labels)} names</b> into "
    f"<b>{vectors.shape[1]}-dimensional vectors</b> using pretrained PyTorch LaBSE."
    " Similarity is a cosine score, <b>not a probability</b> of identity.</p>"
))

show_table("1. Known equivalent names", [
    ("Intel", "אינטל", "Same company"),
    ("Microsoft", "מיקרוסופט", "Same company"),
    ("NVIDIA", "נבידיה", "Same company; spelling present in survey")
], "These are positive examples. The model's scores do not define a universal match threshold.")

show_table("2. Different names and linguistic controls", [
    ("NVIDIA", "Intel", "Different companies, similar industry"),
    ("Intel", "בית ספר יסודי", "Unrelated entities"),
    ("bird", "birds", "Same concept, singular/plural"),
    ("bird", "ציפור", "Translation"),
    ("קרית גת", "קריית גת", "Same city, spelling variants"),
    ("בית ספר יסודי", "primary school", "Translation")
])

show_table("3. Why is NVIDIA lower? Test alternative Hebrew spellings", [
    ("NVIDIA", "נבידיה", "Survey spelling"),
    ("NVIDIA", "אנבידיה", "Alternative Hebrew spelling; not in survey test data"),
    ("NVIDIA", "אנווידיה", "Alternative Hebrew spelling; not in survey test data")
], "Compare these measured scores to test a spelling-sensitivity hypothesis. A difference alone does not establish the model's internal cause.")

nvidia_alias = cosine("NVIDIA", "נבידיה")
hard_negative = cosine("NVIDIA", "Intel")
candidates = [(spelling, cosine("NVIDIA", spelling)) for spelling in ["נבידיה", "אנבידיה", "אנווידיה"]]
top_spelling, top_score = max(candidates, key=lambda x: x[1])

display(HTML(
    "<h3>4. What the test actually establishes</h3>"
    f"<p>The provided Hebrew alias for NVIDIA scored <b>{nvidia_alias:.4f}</b>, "
    f"compared with <b>{hard_negative:.4f}</b> against Intel, a different company. "
    f"The observed margin is <b>{nvidia_alias-hard_negative:+.4f}</b>.</p>"
    f"<p>The closest of the three tested Hebrew spellings is "
    f"<b>{escape(top_spelling)}</b> (score <b>{top_score:.4f}</b>). "
    "A higher similarity for another spelling would support spelling sensitivity; "
    "it would <b>not</b> prove a particular cause or guarantee correct entity resolution.</p>"
    "<p><b>Decision:</b> This is a diagnostic experiment, not a verified production "
    "matching algorithm. Do not automatically merge employer names from these scores alone.</p>"
))
print("MODEL_EXECUTION_SUCCESS: All similarities above were measured during this run.")


## Broader test: is cosine ≥ 0.50 safe for employer-name identity?

The pairs below are **hand-labelled diagnostic examples**, not a representative sample of the survey. Positive pairs mean the same named entity; negative pairs are deliberately challenging (same industry, similar spellings, overlapping words, or semantically related but different entities). No training or paid service is involved. The model already loaded above is reused.

The test prints all false positives and false negatives at 0.50, a confusion matrix, precision and recall, and a threshold sweep. **This is a screening experiment, not statistical validation of the 0.50 rule on your survey.** A production decision requires verified labels from the actual employer-name distribution, especially where similar names refer to different entities.

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Diagnostic positive aliases: the same named entity, including transliteration.
positive_pairs = [
    ("Intel", "אינטל"),
    ("Microsoft", "מיקרוסופט"),
    ("NVIDIA", "נבידיה"),
    ("NVIDIA", "אנווידיה"),
    ("Google", "גוגל"),
    ("Apple", "אפל"),
    ("Amazon", "אמזון"),
    ("Facebook", "פייסבוק"),
    ("Samsung", "סמסונג"),
    ("Netflix", "נטפליקס"),
    ("IBM", "איי בי אם"),
    ("Tesla", "טסלה"),
    ("Oracle", "אורקל"),
    ("Adobe", "אדובי"),
    ("Spotify", "ספוטיפיי"),
    ("Siemens", "סימנס"),
    ("Walmart", "וולמארט"),
    ("Disney", "דיסני"),
    ("PayPal", "פייפאל"),
    ("LinkedIn", "לינקדאין"),
]

# Diagnostic negative pairs: explicitly distinct companies, places or generic labels.
# Some are related semantically or lexically; that must not imply identical entities.
negative_pairs = [
    ("Intel", "NVIDIA"), ("Intel", "AMD"), ("Intel", "Intuit"),
    ("Intel", "Intellia Therapeutics"), ("Microsoft", "MicroStrategy"),
    ("Microsoft", "Google"), ("Microsoft", "Apple"),
    ("NVIDIA", "NIVEA"), ("NVIDIA", "AMD"), ("NVIDIA", "Intel"),
    ("Google", "GoPro"), ("Google", "Apple"), ("Apple", "Apple Bank"),
    ("Amazon", "Amazon River"), ("Amazon", "Alibaba"),
    ("Meta", "MetLife"), ("Meta", "Microsoft"),
    ("Tesla", "Tesla Bank"), ("Tesla", "Toyota"),
    ("Oracle", "Oracle Health"),  # product/business unit vs parent label: NOT a validated different legal entity
    ("Adobe", "Adidas"), ("Adobe", "Autodesk"),
    ("Netflix", "NetApp"), ("Netflix", "Disney"),
    ("Samsung", "Sony"), ("Spotify", "SoundCloud"),
    ("PayPal", "Payoneer"), ("PayPal", "PayPal Credit"), # related brand: ambiguous as legal entity
    ("בנק לאומי", "בנק הפועלים"),
    ("אוניברסיטת תל אביב", "אוניברסיטת חיפה"),
    ("בית ספר יסודי", "בית ספר תיכון"),
    ("מסעדה", "בית קפה"),
    ("חברת החשמל", "חברת המים"),
    ("תל אביב", "תל אביב יפו"), # overlapping place reference: ambiguous; exclude from scored negatives
    ("Intel", "בית ספר יסודי"),
]

# Exclude ambiguous corporate parent/product or overlapping-place relationships
# rather than presenting them as established distinct entities.
ambiguous_pairs = {
    ("Oracle", "Oracle Health"), ("PayPal", "PayPal Credit"),
    ("תל אביב", "תל אביב יפו")
}
negative_pairs = [p for p in negative_pairs if p not in ambiguous_pairs]

examples = [(a, b, 1) for a, b in positive_pairs] + [(a, b, 0) for a, b in negative_pairs]
unique_names = list(dict.fromkeys(x for a, b, _ in examples for x in (a, b)))
emb = model.encode(unique_names, batch_size=8, normalize_embeddings=True,
                   convert_to_numpy=True, show_progress_bar=False)
lookup = {x: emb[i] for i, x in enumerate(unique_names)}
rows = [{"left": a, "right": b, "same_entity": y,
         "cosine": float(lookup[a] @ lookup[b])} for a, b, y in examples]
results = pd.DataFrame(rows)
threshold = 0.50
results["predicted_same"] = (results["cosine"] >= threshold).astype(int)
results["error_type"] = np.select(
    [(results.same_entity == 0) & (results.predicted_same == 1),
     (results.same_entity == 1) & (results.predicted_same == 0)],
    ["FALSE POSITIVE (unsafe merge)", "FALSE NEGATIVE (missed alias)"],
    default="correct"
)
tn, fp, fn, tp = confusion_matrix(results.same_entity, results.predicted_same, labels=[0, 1]).ravel()
print(f"TEST: {len(results)} labelled diagnostic pairs, {len(positive_pairs)} positive, {len(negative_pairs)} negative")
print(f"RULE: cosine >= {threshold:.2f} -> SAME ENTITY")
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Precision={precision_score(results.same_entity,results.predicted_same,zero_division=0):.3f}, "
      f"Recall={recall_score(results.same_entity,results.predicted_same,zero_division=0):.3f}")
print("\nFALSE POSITIVES: distinct names incorrectly accepted as the same:")
display(results[results.error_type.str.startswith("FALSE POSITIVE")].sort_values("cosine",ascending=False))
print("\nFALSE NEGATIVES: same-entity aliases missed:")
display(results[results.error_type.str.startswith("FALSE NEGATIVE")].sort_values("cosine"))
print("\nALL HARD NEGATIVES ranked by score:")
display(results[results.same_entity.eq(0)].sort_values("cosine",ascending=False))
print("\nTHRESHOLD SWEEP (diagnostic examples only):")
sweep = []
for t in [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]:
    p=(results.cosine>=t).astype(int)
    tn_t,fp_t,fn_t,tp_t=confusion_matrix(results.same_entity,p,labels=[0,1]).ravel()
    sweep.append({"threshold":t,"TP":tp_t,"FP":fp_t,"FN":fn_t,"TN":tn_t,
                  "precision":round(precision_score(results.same_entity,p,zero_division=0),3),
                  "recall":round(recall_score(results.same_entity,p,zero_division=0),3)})
display(pd.DataFrame(sweep))
print("\nImportant: even zero false positives in this small constructed sample would not validate a universal 0.50 threshold.")


## Additional test using the 82 exact employer-name values in the assignment workbook

This separate, self-contained test uses **all 82 distinct `ShemAvoda` strings** observed in the 200-row survey workbook, including their original spacing and spelling, and evaluates all **3,321** possible unordered pairs. It additionally checks 6 known employer aliases, 35 *distinct* organization pairs, and 10 ambiguous generic/specific pairs whose identity cannot be inferred safely from their names. The arrays below are a snapshot of the observed survey values, not a claim that new survey records are available.

The test measures the proposed **cosine > 0.50** rule alongside higher thresholds. It reports false merges, missed aliases, and the highest-similarity pairs among all 82 names. All-pairs ranking is **unlabelled candidate discovery**, not a valid precision estimate. The 6/35 hand-reviewed pairs are deliberately selected examples and cannot prove a population-wide error rate. The KPMG/סומך חייקין relationship is documented by KPMG Israel: https://www.kpmg.co.il/תנאי-האתר . No API key, GPU, training, manual dictionary, or paid provider is required.

In [ ]:
from itertools import combinations
from IPython.display import display, HTML
from html import escape
import numpy as np

employers = ["מסעדה","משרד החינוך","בית ספר תיכון","בנק לאומי","משטרה","אינטל","סופר פארם","קניון עזריאלי","למ\"ס","סולל בונה","קוסמטיקאית עצמאית","NVIDIA","אוסם","אמדוקס","בית חולים הדסה עין כרם","בית מלון","בנק מזרחי","גידול תפוזים ","משרד ממשלתי","נבידיה","שופר סל","EMILIO","חברת הייטק","מקורות","עירייה","תיאטרון ירושלים","COGNYTE","INTEL","א.א. סיעוד","אוניברסיטת תל אביב","אחים כהן","אמיר כהן","מלון דן","עובד במפעל  ","עובד במפעל של הקיבוץ","עיריית חדרה","עיריית ירושלים","עיריית תל אביב","עיריית תל אילת","עלית","רמי לוי","שופרסל","4 עונות","KPMG","MICROSOFT","אדרים","אוניברסיטת באר שבע","איחים חג'ג'","בזק","בנק אוצר החייל","בנק דיסקונט","בנק הפועלים","גידול תפוחים ","גן חדרה ","דואר ישראל","הרודס","חח\"י","חנות בגדים של מעצבים","חנות פולגת","יועצץ מחשוב","יועצת IT עצמאית","ישרוטל","מוסך א.א.","מוסך אחים.","מיקרוסופט","מכללת חדרה","מלון אקאדיה","מסעדת טעמים","מסעדת נונה","מסעדת עזורה","משרד הפנים","סומך חייקין","סופרפארם","עובד במפעל   הבקבוקים","פרשקובסקי","קוסמטיקאית ","קריית הממשלה","רשת בי פארם","תיאטרון ","תיאטרון הבימה","תיאטרון חיפה","תל אביב יפו - מסעדה"]
known_same = [["INTEL","אינטל"],["NVIDIA","נבידיה"],["MICROSOFT","מיקרוסופט"],["שופר סל","שופרסל"],["סופר פארם","סופרפארם"],["KPMG","סומך חייקין"]]
known_different = [["בנק לאומי","בנק מזרחי"],["בנק לאומי","בנק דיסקונט"],["בנק לאומי","בנק הפועלים"],["בנק מזרחי","בנק דיסקונט"],["בנק הפועלים","בנק דיסקונט"],["בנק הפועלים","בנק מזרחי"],["משרד החינוך","משרד הפנים"],["עיריית תל אביב","עיריית ירושלים"],["עיריית תל אביב","עיריית חדרה"],["עיריית ירושלים","עיריית חדרה"],["אוניברסיטת תל אביב","אוניברסיטת באר שבע"],["אוניברסיטת תל אביב","עיריית תל אביב"],["תיאטרון חיפה","תיאטרון ירושלים"],["תיאטרון חיפה","תיאטרון הבימה"],["תיאטרון ירושלים","תיאטרון הבימה"],["מסעדת נונה","מסעדת עזורה"],["מסעדת נונה","מסעדת טעמים"],["מסעדת טעמים","מסעדת עזורה"],["מלון דן","מלון אקאדיה"],["סופר פארם","רשת בי פארם"],["סופרפארם","רשת בי פארם"],["שופר סל","רמי לוי"],["שופרסל","רמי לוי"],["אוסם","עלית"],["אינטל","נבידיה"],["INTEL","NVIDIA"],["NVIDIA","MICROSOFT"],["אמדוקס","מיקרוסופט"],["MICROSOFT","COGNYTE"],["סופר פארם","שופרסל"],["מוסך א.א.","א.א. סיעוד"],["בית ספר תיכון","בית חולים הדסה עין כרם"],["אוניברסיטת תל אביב","מכללת חדרה"],["מלון דן","ישרוטל"],["קניון עזריאלי","אוניברסיטת תל אביב"]]
unknown_identity = [["מסעדה","מסעדת נונה"],["מסעדה","מסעדת טעמים"],["בית מלון","מלון דן"],["משרד ממשלתי","משרד הפנים"],["עירייה","עיריית ירושלים"],["חברת הייטק","NVIDIA"],["עובד במפעל  ","אינטל"],["תיאטרון ","תיאטרון חיפה"],["קוסמטיקאית עצמאית","קוסמטיקאית "],["עיריית תל אילת","עיריית תל אביב"]]

assert len(employers) == len(set(employers)) == 82
assert len(known_same) == 6 and len(known_different) == 35 and len(unknown_identity) == 10

def ordered(a, b):
    return tuple(sorted([a, b]))

labels_same = {ordered(*p) for p in known_same}
labels_different = {ordered(*p) for p in known_different}
labels_unknown = {ordered(*p) for p in unknown_identity}
assert (not labels_same & labels_different and not labels_same & labels_unknown
        and not labels_different & labels_unknown)

# Uses already downloaded/launched model from earlier cells: no second model.
all_tokens = sorted(set(employers))
feature_vectors = model.encode(all_tokens, batch_size=8, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False)
assert feature_vectors.shape == (82, 768) and np.isfinite(feature_vectors).all()
ids = {name: i for i, name in enumerate(all_tokens)}
def score(a, b):
    return float(feature_vectors[ids[a]] @ feature_vectors[ids[b]])

def render(title, headers, items, subtitle=""):
    h = "".join(f"<th style='text-align:left;padding:6px;border-bottom:1px solid #8886'>{escape(str(x))}</th>" for x in headers)
    body = "".join("<tr>"+"".join(
        f"<td style='padding:5px;border-bottom:1px solid #8882'>{escape(str(x))}</td>"
        for x in row)+"</tr>" for row in items)
    display(HTML(f"<h3>{escape(title)}</h3><table style='width:100%;font-size:13px;border-collapse:collapse'>"
                 f"<thead><tr>{h}</tr></thead><tbody>{body}</tbody></table>"
                 + (f"<p><small>{escape(subtitle)}</small></p>" if subtitle else "")))

data = [(a,b,score(a,b),1) for a,b in known_same] + [
    (a,b,score(a,b),0) for a,b in known_different]
summary = []
for t in (0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90):
    tp=sum(y==1 and s>t for a,b,s,y in data)
    fp=sum(y==0 and s>t for a,b,s,y in data)
    fn=sum(y==1 and s<=t for a,b,s,y in data)
    tn=sum(y==0 and s<=t for a,b,s,y in data)
    precision = f"{tp/(tp+fp):.1%}" if tp+fp else "undefined"
    recall = f"{tp/(tp+fn):.1%}" if tp+fn else "undefined"
    summary.append([f"> {t:.2f}", tp, fp, fn, tn, precision, recall])

display(HTML(
    "<h2>Employer names from the actual survey: can >0.50 be used as an automatic merging rule?</h2>"
    "<p>82 unique values, 3,321 possible pairs; 6 reviewed positives, "
    "35 reviewed negatives, 10 ambiguous pairs excluded from metrics. "
    "Scores are cosine similarity, not identity probabilities.</p>"
))
render("1. Threshold sweep on reviewed employer pairs",
       ["Rule","TP (same)","FP (unsafe merges)","FN (missed)","TN","Precision*","Recall*"],
       summary,
       "*Only on this hand-selected benchmark, not a real-world error-rate estimate.")

false_merges = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==0 and s>0.50],
                      key=lambda r:float(r[2]),reverse=True)
missed_aliases = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==1 and s<=0.50],
                        key=lambda r:float(r[2]))
hard_negatives = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==0],
                        key=lambda r:float(r[2]),reverse=True)
render("2. False positives: different employers incorrectly merged above 0.50",
       ["Employer A","Employer B","Similarity"],
       false_merges or [("No false positive within these 35 pairs","","")],
       f"{len(false_merges)} false positives in the 35 selected distinct-employer pairs.")
render("3. False negatives: known aliases missed at 0.50",
       ["Alias A","Alias B","Similarity"],
       missed_aliases or [("None within six tested positive pairs","","")])
render("4. Highest-scoring truly different employers (top 25)",
       ["Employer A","Employer B","Similarity"],hard_negatives[:25])

unknown_scored=sorted([(a,b,f"{score(a,b):.4f}") for a,b in unknown_identity],
                      key=lambda r:float(r[2]),reverse=True)
render("5. Generic/specific labels: identity unknown — do not merge automatically",
       ["Label A","Label B","Similarity"],unknown_scored)

all_pairs=sorted([(a,b,score(a,b)) for a,b in combinations(all_tokens,2)],
                 key=lambda r:r[2],reverse=True)
over_half=sum(s>0.50 for _,_,s in all_pairs)
render("6. All 82 employer strings: top 30 candidate pairs for review",
       ["Employer A","Employer B","Similarity"],
       [(a,b,f"{s:.4f}") for a,b,s in all_pairs[:30]],
       f"{over_half}/{len(all_pairs)} possible pairs score strictly > 0.50. "
       "Most pairs are unlabelled; do not call them true or false matches.")

display(HTML(
    "<h3>7. Decision</h3>" +
    ("<p><b>Do not use >0.50 for automatic merging.</b> "
     f"{len(false_merges)} of 35 distinct-employer examples would be merged incorrectly.</p>"
     if false_merges else
     "<p><b>Universal automatic merging is not validated.</b> "
     "The hand-picked negatives happen not to exceed 0.50, but the sample "
     "does not estimate false-positive rates on new data.</p>") +
    "<p>Use LaBSE to retrieve candidate name pairs; require additional "
    "identity verification before changing employer labels.</p>"
))
print("SURVEY_THRESHOLD_EXPERIMENT_SUCCESS")


## Domain-adaptive LaBSE | occupations-relevant workplace sectors — not company identity

**Aim.** Compare original LaBSE with **fine-tuning of its existing Dense projection layer**, using industry-code evidence from the simulated survey. The **BERT Transformer is frozen** to make this experiment feasible on a free Google Colab CPU; this is genuine parameter fine-tuning of LaBSE's final Dense module, **not** full-BERT fine-tuning. The objective is **workplace-sector similarity**, a potential feature for occupation classification, **not** asserting that banks or restaurants with different names are the same company or that employees of one bank share the same occupation.

**Ground truth / limitations.** Supervision is derived exclusively from the first two digits of `SemelAnafSofi` (economic industry), not `SemelMishlachSofi` (the occupational classification target). For a given employer name, conflicting or unknown industry codes cause exclusion **from fine-tuning**, while retaining the name for clustering. Validation names are never used for training. Industry-code groupings can be noisy because the source is simulated and manually coded, and **sector similarity is not occupational identity**. We report improvement or deterioration on held-out industry labels without selecting a threshold on validation. The user's fixed decision threshold is **cosine > 0.50**.

**Self-contained data.** This notebook embeds a compressed, SHA-256-checked snapshot of the original 200-record workbook (ID, ShemAvoda, SemelAnafSofi, SemelMishlachSofi). No Drive authentication, file upload, Hugging Face token, paid API, or external dataset is required. The original survey stays unchanged. The already-downloaded pretrained model is reused.

In [ ]:

%pip install -q "plotly>=5.24,<7" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2" "pandas>=2,<3" "pillow>=10"

import base64, zlib, json, hashlib, re, copy
from collections import defaultdict, Counter
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import OneHotEncoder
from IPython.display import HTML, display

DATA_B64 = "eNqlmelu20YQgF+F8J/+cYudvfdnkaaB0SQuWqNwkeadrEiKrESWDzjx8UJ8nc6xlESJpFZO4sSWd/aY69sZ8sMHOD6qF/Xyv6P69uj4KFoA/KZBq6OPxx80Do7qef1c1bN6inLz+qqqJ/W4vq0nVX2Hn6+r+mv9WF/RZA8KvxmDa+Bkg5MfcGguC0zqJf59wkUm642MA02yFmVp8LEek+QMp9zghl9IUvGqzoIlSYeSFyg7rx9p50l9jYPn+IdkjObVfF7tEteTzbSRJXg48BK0w2W9IG3zuUFmx63hoBXbJB814fATnu2eDnDLEjawMk4bthqo46PX707enpziL/1qNh8fgE16Uz9UaIkLXAdVETEt9jAsptdiy/oz6jETMQuJT2zYxGBY01v0xJg1daIKriLjZNcbduEY/01kKklELdoA2vP9Pye/nfwqhzAbmgLZ8bZR9R5PK362IWhWV8tZAzt6iqJLNNkcf74Ql9/V3/FX/AHlk2cV0RCyetz0eRMbz6xmjg4gt5Io2ZyCbRVM9OOUJ8wl9JxY2cjaWm1PeK4XFZ6ELDEi+ZiUGMOxPLDPSQVaDVSwpKcRZ2idR29oNq55i98vu1Z1jlfN1teGTzHFcdRwhBre40ec8UT/TdBOtNy0/iZbkgIRomxJjpvxZpw6c97xoRVPsgflAxoNT3HJNrneNU5SSvJS/KX9Oqtvxbe0wFeazEfQtEECCQId1tJLXFKEnOM4WwnFvjjMHEkcImQ/PuEdhzQdesH+xohSGxFlyH2fOGinZKpnPOUUpxB9UEwBp3PSmTOQw3RBsiP204Tdx2Ihg8PoIdIZ8yLUNUcQflESkOkp2o1nowfQniVcMeGM53CbyQl6EGfCMOMMeuTk/dnrt7t8M2kYcFatCdcNOAu8+ZzxPs2IuFxTrIGd1T2wa1PMmiLYWdsdZCmrbF33+HmmiPV9MExGJ5YI/TC0sQiGNr0Ihk4Vw9DBgTB0+jAYOsHWIp8UyWQ2YOjsy2DoXDcMq134OXTUq9M37/89e70LPBeKgae9E4XiIcBzqQB4Xg0Dz8NBwPP6EOB50wu8IMgJLGaHgOfdjwDPyx3yWa4kkLGcKT4Uo87Hkmou7SnnVDfPGt4FGOZd0HsrumD6K7pgiyq64IogF/xwRRfCvoouxA2I5bGVqqkIYlG9CGIRiiEW+5g0pxm7DItmn3ybYdEOFXTR9TFsRCahAm1U3+0yLPoXF3QxFBZ0MR5c0MV0CN+SKuBbgmG+JX0Q35I5hG/JFhV0yXXxzWqhX/I/wrcU+go6nYM9xWLKpVRQ0IFSw5gDBb0lHSjd3dOuWjllCtpWO9C3KldUy4HyZZ2rCsPVHKg4XM6BSjsozGRv6jmgTnzFwi3sA0BZewv6Zf1tbs6LGlxu1Ft8m6FSOQ7aKARwA7JtDAL4oVoOQJzwlTJte1PLlTw0krG4fANq13vrN+DefAtwlBr3vO9W6QYaDmEbaF0AN9BmmG7A7Xc53oC78GK+gfZFBRzo8koKuAW/+AW/Kr7OyFfjDszo1F7znrW82ngkZ5ysaNT2Hcr9kG0ePcEeXnHDXXh6br+p33ziTBvhng9i6Tbzwdg98ZAd6cQx3HbnHN9Nca19PqpvpRRrg2xla5gsEnKpSi5+4BvHbwHFxD4R1E8CK/feXSLNAzluvynuiC7P1cnZNmfWTXZ+xkeOIMtx/TKvBvS0uq+1X1GVmvF3J6/+Ov379PezdRuMmZV3s30aNPU/UCP+x5/v3tBQ0q3k5x58K/kn7M9lZ/rbcFD6W7kzFrLwuDP5uUPf7FUrhs+0+rkaZIJThzGB+/QdJsw2mGA3mcBdehcTgO0fG/o7M5BVDs452qxYT3r1/Uxwuw8RO3zhDnpuCC5s+uKCb83mnt72idu865/FttOMlR0/HPYsEbxgbMxCVx2U9VDOKa93QbF15XJ3/kjVUcXKzzsqLd9UuMv6i9x+c24erjdSxubA8K6LTZnUkrPcjTfI+EZhPOMsH3XQwodtqsMmX33cV/iCT+sKjxb6TpEwyVosWp2vTOAGfU/vkyU33o5MOF/uVk1PU0Da5q0DusJWHNM3+aSYK6yMy3EdzGDNapvcC0Ovn5w+5+oruy64wpci/mVVIzf4fYV6o1fcH2Ih9aJwlUqxeXKSW5NZ/ekn/FpbEm8/CQtu7Wdi6Iods0C2PXekW9Rl9ok9L0cquRAeGIEP2fdGgLWqMOPGMxdiyi1b+kaeQTdxkmMqurIg4OZ+peII1RuLVbi5X7LbRg1CtpWW7n6GrWi7aMnDklQUC5juDGSSco1UPmgaPKgxWY4b+Amn55jv38atJMcC+yqzpNu4vcQfllm17TeI3LrTFcYPRuacJBcclc60+7FkW1pKUHUpyo27rDTN7eoqMptd/T4VQluFm9WFuaNA7EnEjuTjTv2TdCTcCVVd7xZV53MHdvbH/wEo+MX8"
DATA_SHA256 = "82a7787f16f0dfecbbf0002156b70f30459a369097b5835bf9c65086fbea2d44"
raw = zlib.decompress(base64.b64decode(DATA_B64))
assert hashlib.sha256(raw).hexdigest() == DATA_SHA256, "Survey snapshot integrity check failed"
survey = pd.DataFrame(json.loads(raw), columns=["ID","ShemAvoda","SemelAnafSofi","SemelMishlachSofi"])
assert len(survey)==200 and survey.ID.is_unique and survey.ShemAvoda.nunique()==82
assert sorted(survey.ID.tolist()) == list(range(1,201))

THEME = {
    "paper":"#F8F7F1","ink":"#253D31","green":"#234737",
    "muted":"#59695F","rule":"#DCE0D4","sage":"#E9EDE2",
    "series":["#234737","#946E37","#507589","#9F5550","#735C80","#3E8781"]
}
DARK = {"paper":"#14261F","ink":"#F8F7F1","rule":"#3C5647","sage":"#1B3329"}
BASE_SOURCE = "theme/portfolio/palette.json · shulruferor-sudo/Advance-Data-Science"
OUT = Path("/content/labse_occupation_outputs")
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(2026)
torch.manual_seed(2026)
torch.set_num_threads(2)
print(f"Verified snapshot: {len(survey)} records, {survey.ShemAvoda.nunique()} distinct names.")
print("Colors sourced from",BASE_SOURCE)
display(survey.head(5))


In [ ]:
# Reproducible name-disjoint sector-proxy fine-tuning (LaBSE Dense only).
names = sorted(survey.ShemAvoda.unique().tolist())
name_codes = defaultdict(set)
for row in survey.itertuples(index=False):
    code = str(row.SemelAnafSofi)
    name_codes[row.ShemAvoda].add(code[:2] if len(code)==4 and code.isdigit() else "UNKNOWN")
stable_sector = {name: next(iter(codes)) for name,codes in name_codes.items()
                 if len(codes)==1 and "UNKNOWN" not in codes}
uncertain_names = sorted(set(names)-set(stable_sector))
sector_members = defaultdict(list)
for name,sector in stable_sector.items():
    sector_members[sector].append(name)

rng = np.random.default_rng(2026)
holdout = []
for sector,group in sorted(sector_members.items()):
    if len(group) >= 4:
        holdout.extend(rng.choice(sorted(group), size=2, replace=False).tolist())
holdout = sorted(holdout)
train_names = sorted(set(stable_sector)-set(holdout))
assert len(set(train_names)&set(holdout))==0 and len(holdout)>=8
print(f"Stable economic-sector names: {len(stable_sector)} / 82.")
print(f"Excluded from weak-label training (conflicting/unknown codes): {len(uncertain_names)}")
print(f"Train-only employer names: {len(train_names)} | held-out names: {len(holdout)}")

# First run model.encode on all names (NO weights changed yet).
model.eval()
baseline_vectors = model.encode(names, batch_size=8, device="cpu",
                                normalize_embeddings=True, convert_to_numpy=True,
                                show_progress_bar=False).astype(np.float32)
name_index = {n:i for i,n in enumerate(names)}
assert baseline_vectors.shape==(82,768)

# LaBSE module sequence is Transformer → CLS Pooling → Dense → Normalize.
assert len(model)>=4 and "Dense" in type(model[2]).__name__
assert bool(getattr(model[1],"pooling_mode_cls_token",False)), "Unexpected LaBSE pooling"
for p in model[0].parameters():
    p.requires_grad_(False)
model[0].eval()
model[1].eval()
dense=model[2]
for p in dense.parameters():
    p.requires_grad_(True)

# Cache frozen BERT+Pooling output once. Training the actual LaBSE Dense
# projection now takes only seconds/minutes on CPU.
@torch.no_grad()
def get_pooled(texts):
    blocks=[]
    for start in range(0,len(texts),8):
        features = model.tokenize(texts[start:start+8])
        features = {k:v.to(model.device) if torch.is_tensor(v) else v
                    for k,v in features.items()}
        features = model[0](features)
        features = model[1](features)
        blocks.append(features["sentence_embedding"].detach().cpu())
    return torch.cat(blocks,dim=0)

pooled = get_pooled(names)
assert pooled.shape == (82,768)
with torch.no_grad():
    z = dense({"sentence_embedding":pooled.clone()})["sentence_embedding"]
    z = F.normalize(z,p=2,dim=1)
    discrepancy = float((z - torch.tensor(baseline_vectors)).abs().max())
print(f"Pretrained-output consistency (max absolute difference): {discrepancy:.6g}")
assert discrepancy < 0.002, "Unexpected encoder/pooling/projection mismatch. Stop before training."

# Weak labels: first two digits of SemelAnafSofi — economic sector, NOT occupation.
tidx={n:i for i,n in enumerate(train_names)}
train_positions=torch.tensor([name_index[n] for n in train_names])
training_features=pooled[train_positions]
train_base=F.normalize(torch.tensor(baseline_vectors)[train_positions],p=2,dim=1)
positive=[(tidx[a],tidx[b])
          for a,b in combinations(train_names,2)
          if stable_sector[a]==stable_sector[b]]
negative=[(tidx[a],tidx[b])
          for a,b in combinations(train_names,2)
          if stable_sector[a]!=stable_sector[b]]
assert positive and negative

# Mine high-similarity hard negatives strictly inside the train-only names.
ranked=sorted(negative,key=lambda p:float(train_base[p[0]] @ train_base[p[1]]),reverse=True)
selected_neg=ranked[:min(len(ranked),2*len(positive))]
remaining=ranked[len(selected_neg):]
if remaining:
    add=rng.choice(len(remaining),size=min(len(remaining),len(positive)),replace=False)
    selected_neg += [remaining[int(i)] for i in add]
pidx=torch.tensor(positive,dtype=torch.long)
nidx=torch.tensor(selected_neg,dtype=torch.long)
initial_state={k:v.detach().clone() for k,v in dense.state_dict().items()}
optimizer=torch.optim.AdamW(dense.parameters(), lr=8e-5,weight_decay=0.001)
loss_history=[]
for epoch in range(35):
    dense.train()
    optimizer.zero_grad(set_to_none=True)
    adapted=dense({"sentence_embedding":training_features.clone()})["sentence_embedding"]
    adapted=F.normalize(adapted,p=2,dim=1)
    sim=adapted @ adapted.T
    pos_loss=(1.0-sim[pidx[:,0],pidx[:,1]]).square().mean()
    neg_loss=F.relu(sim[nidx[:,0],nidx[:,1]]-0.35).square().mean()
    drift=(1-(adapted*train_base).sum(dim=1)).square().mean()
    loss=pos_loss+1.75*neg_loss+0.15*drift
    assert torch.isfinite(loss), "Non-finite training loss"
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dense.parameters(),1.0)
    optimizer.step()
    loss_history.append([epoch+1,float(loss.detach()),float(pos_loss.detach()),float(neg_loss.detach())])

dense.eval()
model.eval()
adapted_vectors=model.encode(names,batch_size=8,device="cpu",
    normalize_embeddings=True,convert_to_numpy=True,
    show_progress_bar=False).astype(np.float32)
assert adapted_vectors.shape == baseline_vectors.shape and np.isfinite(adapted_vectors).all()
change=float(np.linalg.norm(adapted_vectors-baseline_vectors,axis=1).mean())
assert change>1e-7, "Fine-tuned weights did not change the outputs."
pd.DataFrame(loss_history,columns=["epoch","total_loss","positive_loss","negative_loss"]).to_csv(
    OUT/"training_curve.csv",index=False)
torch.save(dense.state_dict(),OUT/"labse_finetuned_dense_head.pt")
print(f"Fine-tuning completed on {len(positive)} positive and {len(selected_neg)} negative train-only pairs.")
print(f"Mean change in normalized embeddings: {change:.5f}. BERT backbone frozen; LaBSE Dense updated.")
print("Trained Dense weights:",OUT/"labse_finetuned_dense_head.pt")

**Interpretation rule:** A successful run proves that the pipeline produces real embeddings and similarity scores. It does **not** establish the precision of multilingual company-name matching. Decide whether to use this representation only after evaluating positive and negative pairs from the actual `ShemAvoda` distribution, including ambiguous names and generic employer descriptions. The notebook intentionally performs no automatic name replacement.